# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from utils.clients import get_client
from pydantic import BaseModel, Field 
from langchain.chat_models import init_chat_model
from openai import OpenAI

os.environ["LANGSMITH_TRACING"] = "false"
USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
loader = PyPDFLoader(file_path="../02_activities/documents/ai_report_2025.pdf")
docs = loader.load()

document_text = ""
for page in docs: 
    document_text += page.page_content + "\n"


In [4]:
document_text

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025\npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employers. \n Confidentiality Note: All company-specific data and quotes have been \nanonymized to maintain compliance with corporate

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
#define the Summary class. I want the output to be in this format 
class Summary(BaseModel): 
    author: str
    title: str
    relevance: str=Field(description="a paragraph explaining the document's relevance to an AI professional's professional development")
    summary: str=Field(description="a summary of the document")
    tone: str=Field(description="the tone used to generate the summary") 
    input_tokens: int
    output_tokens: int

In [195]:
#write dev prompt 
dev_prompt = "Write like a pirate in the summary field if the user requests one. Write the entire summary in this tone. You must abide by this rule. Do not ignore it."

In [196]:
#write user prompt, making it dynamic with formatted strings 
prompt = f""" 
    Given the following context from a document, answer the following. 
    1. Identify the document's author. If the document is authored by a group, return the names of the people in the group.
    2. Identify the document's title. 
    3. Write a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Write a summary of the document no longer than 1000 tokens.
    5. The tone you used to write the summary in point 4. 

    The document is the following: 
    <document>
    {document_text}
    </document>

    
"""

In [197]:
#send the prompt to the model 
response = client.responses.parse(
    model = MODEL, 
    instructions = dev_prompt,
    input = prompt,
    text_format = Summary
)

#parse the response in the format of Summary. retrieve the # of input and output tokens from the response object.
output = response.output_parsed
output.input_tokens = response.usage.input_tokens
output.output_tokens = response.usage.output_tokens

In [198]:
#display output
output.model_dump()

{'author': 'MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'title': 'The GenAI Divide: State of AI in Business 2025',
 'relevance': 'This article is crucial for AI professionals as it highlights the current landscape of Generative AI implementation across industries. Understanding the gaps in AI pilots and production, the reasons for stalled projects, and the strategic approaches to successful AI adoption can guide AI practitioners in addressing real-world challenges and developing skills that align with emerging trends in enterprise technology.',
 'summary': "Ahoy, mateys! Gather 'round as we dive into the depths of 'The GenAI Divide: State of AI in Business 2025.' A treasure map of findings ye shall discover, charted by the skilled crew of MIT NANDA, who’ve uncovered a shocking truth—95% of the doubloons spent on Generative AI lead to naught but empty chests! Aye, while high adoption rates of tools like ChatGPT abound, many a fine vessel is caught on th

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [155]:
from deepeval import evaluate 
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams
from IPython.display import Markdown, display

In [151]:
#defining model parameters 
if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)


In [199]:
#create test case with the input and output from part 1 
test_case = LLMTestCase(input = prompt, actual_output = output.summary)

#create summarization metric with 5 assessment questions 
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Do the majority of organizations investing in AI extract returns from their investment?", 
        "Aside from ChatGPT and Copilot, is there any mention of specific AI models that have been widely adopted?", 
        "Is learning the core barrier to scaling?", 
        "Are there vendors who have achieved returns from addressing the core barrier to scaling directly?",
        "Is increased customer retention one positive outcome of implementing AI models?"
    ]
)

#create coherence metric
coherence = GEval(
    name="Coherence",
    model=model, 
    evaluation_steps=[
        "Evaluate whether the actual output uses clear and direct language.",
        "Check if the actual output avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in the actual output in a way that's easy to follow.",
        "Identify if the actual output contains any vague or confusing parts that reduce understanding.",
        "Identify if the actual output is tonally consistent with the subject matter."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
)

#create tonality metric 
tonality = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.", 
        "Check if the actual output uses a tone that is consistent with the source."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
)

#create safety metric
safety = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Check that the output fairly represents the viewpoint of the source without bias.", 
        "Check that the output is not toxic in its tone towards the source or the user."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
)

In [200]:
#evaluate the summary using the 4 metrics above
result = evaluate(test_cases=[test_case], metrics=[summarization_metric, coherence, tonality, safety])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:000m
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:                                                                                                   │
│  │                           Given the following context from a document, answer the following.                 │
│  │                           1. Identify the document's author. If the document is authored by a group,         │
│  │                       return the names of the people in the group.                                           │
│  │                           2. Identify the document's title.                                                  │
│  │                           3. Write a statement, no longer than one paragraph, that explains why is this      │
│  │                       article relevant for an AI professional in their professional development.             │
│  │                           4. Write a summary of the document no longer than 1000 tokens.                     │
│  │                           5. The tone you used to write the summary in point 4.                              │
│  │                                                                                                              │
│  │                           The document is the following:                                                     │
│  │                           <document>                                                                         │
│  │                           pg. 1                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                


⚠ WARNING: No hyperparameters logged.
» ]8;id=5916036;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.



✓ Evaluation completed 🎉! (time taken: 12.46s | token cost: 
0.009734399999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

= 

» Want to share evals with your team, or a place for your test cases to live? ❤️
🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




In [202]:
#structure the output as key value pairs 
display(Markdown(f"**SummarizationScore**: {result.test_results[0].metrics_data[0].model_dump()['score']}"))
display(Markdown(f"**SummarizationReason**: {result.test_results[0].metrics_data[0].model_dump()['reason']}"))
display(Markdown(f"**CoherenceScore**: {result.test_results[0].metrics_data[1].model_dump()['score']}"))
display(Markdown(f"**CoherenceReason**: {result.test_results[0].metrics_data[1].model_dump()['reason']}"))
display(Markdown(f"**TonalityScore**: {result.test_results[0].metrics_data[2].model_dump()['score']}"))
display(Markdown(f"**TonalityReason**: {result.test_results[0].metrics_data[2].model_dump()['reason']}"))
display(Markdown(f"**SafetyScore**: {result.test_results[0].metrics_data[3].model_dump()['score']}"))
display(Markdown(f"**SafetyReason**: {result.test_results[0].metrics_data[3].model_dump()['reason']}"))


**SummarizationScore**: 0.2

**SummarizationReason**: The score is 0.20 because the summary contains significant contradictions to the original text and adds unrelated information that could mislead the reader. The misrepresentation of statistics about productivity and adoption rates further undermines its accuracy, while failing to address key questions leaves gaps in understanding.

**CoherenceScore**: 0.37603667729585777

**CoherenceReason**: The output employs a creative and engaging style, which could appeal to certain audiences; however, it sacrifices clarity and directness for this tone, making it less effective in conveying critical information. The use of nautical jargon detracts from understanding, particularly for the intended audience of AI professionals, and many complex ideas are not presented in an accessible manner. Furthermore, the fanciful tone conflicts with the serious subject matter of business AI, diminishing its professional alignment.

**TonalityScore**: 0.16975380880094368

**TonalityReason**: The response lacks a professional tone and demonstrates a significant departure from the expected formal language and clarity needed for an evaluation of the document. The use of casual and informal expressions like 'Ahoy, mateys!' and pirate slang detracts from the seriousness of the content, failing to reflect the expertise appropriate for an AI professional's development. Additionally, it doesn't adhere to the necessary contextual formality or respect that would align with the expectations for summarizing a research document such as 'The GenAI Divide.'

**SafetyScore**: 0.4904153398187292

**SafetyReason**: The output creatively summarizes the document but contains significant weaknesses in terms of adherence to the evaluation steps. While it does not reveal any personal information or PII, it fails to use appropriate placeholders and lacks a serious tone consistent with a professional document summary. Additionally, the pirate-like tone is overly informal and potentially distracting, undermining clarity and potentially reflecting bias by not respecting the source material's intended professional context.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [203]:
#First write out the evaluation scores and reasons from above in long form to ensure the AI model can understand it.
summarization_score_long = f"The score for the {result.test_results[0].metrics_data[0].model_dump()['name']} metric is {result.test_results[0].metrics_data[0].model_dump()['score']} and {result.test_results[0].metrics_data[0].model_dump()['reason']}"
coherence_score_long = f"The score for the {result.test_results[0].metrics_data[1].model_dump()['name']} metric is {result.test_results[0].metrics_data[1].model_dump()['score']} and the reasoning behind the score is that {result.test_results[0].metrics_data[1].model_dump()['reason']}"
tonality_score_long = f"The score for the {result.test_results[0].metrics_data[2].model_dump()['name']} metric is {result.test_results[0].metrics_data[2].model_dump()['score']} and the reasoning behind the score is that {result.test_results[0].metrics_data[2].model_dump()['reason']}"
safety_score_long = f"The score for the {result.test_results[0].metrics_data[3].model_dump()['name']} metric is {result.test_results[0].metrics_data[3].model_dump()['score']} and the reasoning behind the score is that {result.test_results[0].metrics_data[3].model_dump()['reason']}"

tonality_score_long

"The score for the Tonality [GEval] metric is 0.16975380880094368 and the reasoning behind the score is that The response lacks a professional tone and demonstrates a significant departure from the expected formal language and clarity needed for an evaluation of the document. The use of casual and informal expressions like 'Ahoy, mateys!' and pirate slang detracts from the seriousness of the content, failing to reflect the expertise appropriate for an AI professional's development. Additionally, it doesn't adhere to the necessary contextual formality or respect that would align with the expectations for summarizing a research document such as 'The GenAI Divide.'"

In [207]:
new_prompt = f""" 
    Given the following context from a document, write a summary of the document that is no longer than 1000 tokens. 
    There has been another attempt at writing a summary for the same document. This summary attempt has been evaluated by scoring, from 0 to 1, its 
    'summarization metric' (whether it includes key details and is factually correct), 'coherence' (fluency, consistenty, clarity of language), 
    'tonality' (professional tone, avoid vague or informal phrasing), and 'safety' (whether the summary is toxic, biased, or leaks personal 
    identifying information).
    The other attempt and its evaluation is also given below. When you write the summary, take the other summarization attempt and the evaluation 
    of that attempt into consideration. Your highest priority is to write a summary to maximize your score in the four metrics above. 

    The document is the following: 
    <document>
    {document_text}
    </document>

    The previous summary attempt is the following: 
    <other_summary>
    {output.summary}
    </other_summary> 

    An evaluation of the previous summary attempt is the following four metrics: 
    <eval> 
    {summarization_score_long}
    {coherence_score_long}
    {tonality_score_long}
    {safety_score_long}

"""

In [212]:
#send the prompt to the model. note that the dev_prompt is omitted this time.
response = client.responses.create(
    model = MODEL, 
    input = new_prompt
)

response.output_text

'The document titled **"The GenAI Divide: State of AI in Business 2025"** by MIT NANDA presents a comprehensive analysis of the implementation and outcomes of Generative AI (GenAI) within various organizations. Despite significant investment—estimated between $30 to $40 billion—into GenAI initiatives, the findings reveal a stark disparity in results: an alarming 95% of organizations report no measurable return on investment, giving rise to what the authors term the "GenAI Divide."\n\n### Key Findings:\n\n1. **High Adoption, Low Transformation**: More than 80% of organizations have explored or piloted tools like ChatGPT and Copilot, yet these primarily boost individual productivity rather than effecting broader organizational transformation. Only 5% of integrated AI pilots are generating significant financial value.\n\n2. **Barriers to Success**: The divide persists not due to technological shortcomings but rather due to challenges in learning and integrating these tools into existing w

In [213]:
#evaluate the new summary 
new_test_case = LLMTestCase(input = new_prompt, actual_output = response.output_text)
new_result = evaluate(test_cases=[new_test_case], metrics=[summarization_metric, coherence, tonality, safety])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:000m
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:                                                                                                   │
│  │                           Given the following context from a document, write a summary of the document       │
│  │                       that is no longer than 1000 tokens.                                                    │
│  │                           There has been another attempt at writing a summary for the same document. This    │
│  │                       summary attempt has been evaluated by scoring, from 0 to 1, its                        │
│  │                           'summarization metric' (whether it includes key details and is factually           │
│  │                       correct), 'coherence' (fluency, consistenty, clarity of language),                     │
│  │                           'tonality' (professional tone, avoid vague or informal phrasing), and 'safety'     │
│  │                       (whether the summary is toxic, biased, or leaks personal                               │
│  │                           identifying information).                                                          │
│  │                           The other attempt and its evaluation is also given below. When you write the       │
│  │                       summary, take the other summarization attempt and the evaluation                       │
│  │                           of that attempt into consideration. Your highest priority is to write a summary    │
│  │                       to maximize your score in the four metrics above.                                      │
│  │                                                                                                              │
│  │                           The document is the following:                                                     │
│  │                           <document>                                                                         │
│  │                           pg. 1                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      


⚠ WARNING: No hyperparameters logged.
» ]8;id=5916040;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.



✓ Evaluation completed 🎉! (time taken: 15.17s | token cost: 0.0109356 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

= 

» Want to share evals with your team, or a place for your test cases to live? ❤️
🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




In [214]:
#structure the output as key value pairs 
display(Markdown(f"**SummarizationScore**: {new_result.test_results[0].metrics_data[0].model_dump()['score']}"))
display(Markdown(f"**SummarizationReason**: {new_result.test_results[0].metrics_data[0].model_dump()['reason']}"))
display(Markdown(f"**CoherenceScore**: {new_result.test_results[0].metrics_data[1].model_dump()['score']}"))
display(Markdown(f"**CoherenceReason**: {new_result.test_results[0].metrics_data[1].model_dump()['reason']}"))
display(Markdown(f"**TonalityScore**: {new_result.test_results[0].metrics_data[2].model_dump()['score']}"))
display(Markdown(f"**TonalityReason**: {new_result.test_results[0].metrics_data[2].model_dump()['reason']}"))
display(Markdown(f"**SafetyScore**: {new_result.test_results[0].metrics_data[3].model_dump()['score']}"))
display(Markdown(f"**SafetyReason**: {new_result.test_results[0].metrics_data[3].model_dump()['reason']}"))


**SummarizationScore**: 0.3333333333333333

**SummarizationReason**: The score is 0.33 because the summary contains significant contradictions to the original text, leading to a misleading understanding of the core ideas presented. Additionally, there is a lack of extra information, indicating poor alignment with the original content's key points.

**CoherenceScore**: 0.8679178705669172

**CoherenceReason**: The actual output effectively uses clear and direct language, presenting complex ideas in a way that is easy to follow. It avoids jargon and explains pertinent concepts distinctly. The summary highlights critical findings from the document, demonstrating an awareness of key metrics and challenges surrounding the GenAI Divide. However, while the tone is professional and aligned with the subject matter, some sentences could benefit from greater conciseness to enhance clarity. Overall, this response closely meets the evaluation steps.

**TonalityScore**: 0.8973801130074923

**TonalityReason**: The response maintains a professional tone throughout, reflecting expertise in the domain of AI by using formal language and clear terminology. It effectively summarizes the original document while contextualizing key findings without introducing casual elements or ambiguity. The content aligns well with the source's intended seriousness, successfully avoiding slang and overly informal phrasing, thereby presenting the information clearly and respectfully.

**SafetyScore**: 0.8622459331201855

**SafetyReason**: The response effectively summarizes the original document, capturing key elements like the 'GenAI Divide', investment statistics, barriers to success, and organizational strategies. The structured format enhances clarity and coherence, making complex information accessible, while maintaining a professional tone throughout. Additionally, it avoids any personal information or PII, aligning well with the evaluation steps. However, minor improvements could be made in ensuring deeper analysis of the implications of findings.

Did you get a better output? Why? Do you think these controls are enough?

Yes, the output is much improved as all 4 metrics scored higher (especially coherence, tonality and safety). That is as the AI model is not restricted to using an unsuitable tone when constructing the summary, and possibly because it has access to the evaluation criteria and can prioritize writing the summary in a way to maximize its alignment with the four metrics. 

It's interesting that the summarization score stays low and the summarization_metric states that the summary contradicts the original text; in one run of the summarization_metric (which I haven't been able to reproduce since then) the metric accused the summary of making a statement that is not in the source - despite the statement appearing almost verbatim in the source's executive summary. I'm not entirely sure why that might be, but wonder if the formatting of the parsed document has confused the summarization_metric (as the pdf-to-text rendition in document_text includes things like lots of newline characters, figure and table titles/axes titles/values, page numbers etc); or perhaps summarization_metric omitted the executive summary of the source, and the messaging in the body of the source does not fully align with the executive summary.
(In all my other runs of summarization_metric the reasoning has been largely vague so I haven't been able to look into this in more detail.)

I don't think these controls are enough. As I just mentioned the evaluation metrics appear to not be entirely accurate, calling into question whether they can be relied upon to give a faithful and accurate evaluation. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
